# SageMaker AI MLflow — SFT Fine-Tuning a Smaller Model

In this notebook, you will fine-tune a smaller **Qwen2.5-7B** model using the curated dataset from the previous notebook (05-1). This is a practical example of **knowledge distillation** — transferring the capabilities of a large model into a smaller, more cost-efficient one.

**What you will learn:**
- Run serverless fine-tuning with `SFTTrainer` (LoRA) on Qwen2.5-7B Instruct
- Deploy the fine-tuned model with DJL LMI + vLLM on a SageMaker endpoint
- Test the fine-tuned 7B model on medical triage queries

### Why Fine-Tune a Smaller Model?

| Aspect | Qwen3-8B (Agent) | Qwen2.5-7B Fine-Tuned (this notebook) |
|---|---|---|
| **Parameters** | 8 billion | 7 billion |
| **Instance** | ml.g5.2xlarge | ml.g5.2xlarge |
| **Domain knowledge** | General + prompted via tools | Specialized via SFT on curated traces |

### Prerequisites
- Completed the dataset curation notebook (05-1) with dataset registered in SageMaker AI Registry
- A SageMaker AI MLflow App
- Service quota for serverless fine-tuning and `ml.g5.2xlarge` for deployment

<div style="padding: 15px; background-color: #fff3cd; border-left: 5px solid #ffc107; color: #856404;">
<strong>⚠️ Important:</strong> The cell below installs libraries and restarts the kernel. After the restart, continue with the next cell.
</div>

In [ ]:
# Install SageMaker SDK v3 and dependencies
%pip install "sagemaker>=3.5.0" mlflow==3.9.0 sagemaker-mlflow==0.2.0 datasets==4.5.0 --quiet

# restart kernel
import IPython
IPython.Application.instance().kernel.do_shutdown(True) #automatically restarts kernel

## 1. Environment Setup

In [ ]:
import json
import os
import boto3
import mlflow
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix
region = sess.boto_region_name
s3_client = boto3.client("s3")

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

# Retrieve variables stored from previous notebook
%store -r

print(f"Role: {role}")
print(f"Bucket: {bucket_name}")
print(f"Region: {region}")
print(f"Train dataset: {train_s3_path}")
print(f"Val dataset: {val_s3_path}")

## 2. Configure SageMaker AI MLflow

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(ft_experiment_name)

print(f"MLflow tracking server: {mlflow.get_tracking_uri()}")
print(f"Fine-tuning experiment: {ft_experiment_name}")

## 3. Load Registered Datasets

We retrieve the datasets registered in the SageMaker AI Registry from the previous notebook.

In [ ]:
from sagemaker.ai_registry.dataset import DataSet
from sagemaker.ai_registry.dataset_utils import CustomizationTechnique

# Retrieve the registered datasets
dataset_train = DataSet.get(name="medical-triage-sft-train")
dataset_val = DataSet.get(name="medical-triage-sft-val")

print(f"Training dataset ARN: {dataset_train.arn}")
print(f"Validation dataset ARN: {dataset_val.arn}")

## 4. Serverless Fine-Tuning with SFTTrainer

We use the SageMaker `SFTTrainer` to run serverless Supervised Fine-Tuning with LoRA on **Qwen2.5-7B Instruct**. This is a serverless job — no training infrastructure to manage.

The trainer:
1. Loads the base model from JumpStart
2. Applies LoRA adapters for parameter-efficient fine-tuning
3. Trains on the curated dataset
4. Merges the LoRA weights and registers the model in a Model Package Group
5. Tracks metrics in MLflow automatically

In [ ]:
base_model_id = "huggingface-llm-qwen2-5-7b-instruct"

if default_prefix:
    output_path = f"s3://{bucket_name}/{default_prefix}/{base_model_id}"
else:
    output_path = f"s3://{bucket_name}/{base_model_id}"

# Set MLflow endpoint for training job integration
os.environ["SAGEMAKER_MLFLOW_CUSTOM_ENDPOINT"] = (
    f"https://mlflow.sagemaker.{region}.app.aws"
)

print(f"Base model: {base_model_id}")
print(f"Output path: {output_path}")

### Create Model Package Group

A Model Package Group organizes versioned model artifacts from fine-tuning runs.

In [ ]:
from botocore.exceptions import ClientError
from sagemaker.core.resources import ModelPackageGroup

model_package_group_name = f"{base_model_id}-medical-triage-mpg"

try:
    model_package_group = ModelPackageGroup.get(
        model_package_group_name=model_package_group_name
    )
    print(f"Model Package Group already exists: {model_package_group_name}")
except ClientError:
    model_package_group = ModelPackageGroup.create(
        model_package_group_name=model_package_group_name,
        model_package_group_description="Medical triage SFT models fine-tuned from Qwen3-8B curated traces",
    )
    print(f"Created Model Package Group: {model_package_group_name}")

### Configure and Launch SFT Training Job

In [ ]:
from sagemaker.train.common import TrainingType
from sagemaker.train.sft_trainer import SFTTrainer

trainer = SFTTrainer(
    model=base_model_id,
    training_type=TrainingType.LORA,
    model_package_group=model_package_group_name,
    training_dataset=dataset_train,
    validation_dataset=dataset_val,
    s3_output_path=output_path,
    sagemaker_session=sess,
    role=role,
    accept_eula=True,
)

print("SFTTrainer configured.")

In [ ]:
# View default hyperparameters
print("Default hyperparameters:")
for k, v in trainer.hyperparameters.to_dict().items():
    print(f"  {k}: {v}")

In [ ]:
# Override hyperparameters for our small dataset
trainer.hyperparameters.learning_rate = 0.0001
trainer.hyperparameters.global_batch_size = 8
trainer.hyperparameters.max_epochs = 3
trainer.hyperparameters.lr_warmup_ratio = 0.1

print("Modified hyperparameters:")
for k, v in trainer.hyperparameters.to_dict().items():
    print(f"  {k}: {v}")

<div style="padding: 15px; background-color: #d1ecf1; border-left: 5px solid #17a2b8; color: #0c5460;">
<strong>ℹ️ Note:</strong> The serverless fine-tuning job typically takes 15-30 minutes depending on dataset size. Training metrics are automatically tracked in MLflow.
</div>

In [ ]:
# Launch the training job
training_job = trainer.train(wait=True)

TRAINING_JOB_NAME = training_job.training_job_name
print(f"Training job completed: {TRAINING_JOB_NAME}")

## 5. Deploy the Fine-Tuned Model

We deploy the fine-tuned Qwen2.5-7B model using DJL LMI with vLLM on a `ml.g5.2xlarge` instance — a much smaller (and cheaper) instance than the one used for the 8B teacher model.

### Get the Merged Model Artifact

In [ ]:
from sagemaker.core.resources import ModelPackage, ModelPackageGroup
from sagemaker.core import s3

model_package_version = "1"

model_package_group = ModelPackageGroup.get(model_package_group_name)
fine_tuned_model_package_arn = (
    f"{model_package_group.model_package_group_arn.replace('model-package-group', 'model-package', 1)}/{model_package_version}"
)

print(f"Model Package ARN: {fine_tuned_model_package_arn}")

model_package = ModelPackage.get(fine_tuned_model_package_arn)

# Get the merged model artifact S3 URI
merged_model_s3_uri = (
    s3.s3_path_join(
        model_package.inference_specification.containers[
            0
        ].model_data_source.s3_data_source.s3_uri,
        "checkpoints",
        "hf_merged",
    )
    + "/"
)
print(f"Merged model S3 URI: {merged_model_s3_uri}")

### Create Endpoint and Deploy

In [ ]:
model_name = f"{base_model_id}-medical-triage-sft"
endpoint_config_name = f"{base_model_id}-medical-triage-config"
endpoint_name = "workshop-qwen25-7b-sft"

instance_type = "ml.g5.2xlarge"
health_check_timeout = 700

print(f"Endpoint name: {endpoint_name}")
print(f"Instance type: {instance_type}")

In [ ]:
# Get the DJL LMI inference container image
CONTAINER_VERSION = "0.36.0-lmi18.0.0-cu128"
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/djl-inference:{CONTAINER_VERSION}"

env = {
    "HF_MODEL_ID": "/opt/ml/model",
    "OPTION_TRUST_REMOTE_CODE": "true",
    "OPTION_MODEL_LOADING_TIMEOUT": "3600",
    "OPTION_TENSOR_PARALLEL_DEGREE": "max",
    "SERVING_FAIL_FAST": "true",
    "OPTION_ROLLING_BATCH": "disable",
    "OPTION_ASYNC_MODE": "true",
    "OPTION_ENTRYPOINT": "djl_python.lmi_vllm.vllm_async_service",
    "OPTION_DTYPE": "bf16",
    "OPTION_QUANTIZE": "fp8",
    "OPTION_MAX_MODEL_LEN": json.dumps(1024 * 8),
}

print(f"Inference image: {inference_image}")

In [ ]:
from sagemaker.core.resources import Model
from sagemaker.core.shapes import (
    ContainerDefinition,
    ModelDataSource,
    S3ModelDataSource,
)

fine_tuned_model = Model.create(
    model_name=model_name,
    primary_container=ContainerDefinition(
        image=inference_image,
        model_data_source=ModelDataSource(
            s3_data_source=S3ModelDataSource(
                s3_uri=merged_model_s3_uri,
                s3_data_type="S3Prefix",
                compression_type="None",
            )
        ),
        environment=env,
    ),
    execution_role_arn=role,
)

print(f"Model created: {model_name}")

In [ ]:
from sagemaker.core.resources import Endpoint, EndpointConfig
from sagemaker.core.shapes import ProductionVariant

print(f"Creating EndpointConfig: {endpoint_config_name}")
endpoint_config = EndpointConfig.create(
    endpoint_config_name=endpoint_config_name,
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            model_name=model_name,
            instance_type=instance_type,
            initial_instance_count=1,
            model_data_download_timeout_in_seconds=health_check_timeout,
        )
    ],
)

print(f"Creating Endpoint: {endpoint_name}")
endpoint = Endpoint.create(
    endpoint_name=endpoint_name, endpoint_config_name=endpoint_config_name
)
endpoint.wait_for_status("InService")
print(f"Endpoint {endpoint_name} is InService")

## 6. Test: Compare SFT Model vs Teacher Model

Compare the SFT fine-tuned 7B model against the teacher 8B model (Qwen3-8B) on the same medical triage queries.
The fine-tuned model should produce comparable quality despite being smaller.

In [ ]:
import io

sagemaker_runtime = boto3.client("sagemaker-runtime")
TEACHER_ENDPOINT = "workshop-qwen3-8b"

class LineIterator:
    def __init__(self, stream):
        self.byte_iterator = iter(stream)
        self.buffer = io.BytesIO()
        self.read_pos = 0
    def __iter__(self):
        return self
    def __next__(self):
        while True:
            self.buffer.seek(self.read_pos)
            line = self.buffer.readline()
            if line and line[-1] == ord("\n"):
                self.read_pos += len(line)
                return line[:-1]
            try:
                chunk = next(self.byte_iterator)
            except StopIteration:
                if self.read_pos < self.buffer.getbuffer().nbytes:
                    continue
                raise
            if "PayloadPart" not in chunk:
                continue
            self.buffer.seek(0, io.SEEK_END)
            self.buffer.write(chunk["PayloadPart"]["Bytes"])

def parse_streaming_response(line_str):
    if not line_str.strip() or line_str.strip() == "data: [DONE]":
        return None
    if line_str.startswith("data: "):
        line_str = line_str[6:]
    try:
        data = json.loads(line_str)
        if "choices" in data:
            for choice in data["choices"]:
                if "delta" in choice and "content" in choice["delta"]:
                    return choice["delta"]["content"]
    except json.JSONDecodeError:
        pass
    return None

def invoke_fine_tuned_model(prompt):
    """Invoke the SFT fine-tuned model with streaming."""
    request_body = {
        "messages": [
            {"role": "user", "content": [{"type": "text", "text": prompt}]}
        ],
        "max_tokens": 1024,
        "temperature": 0.3,
        "top_p": 0.9,
        "stop": ["<|im_end|>"],
        "stream": True,
    }
    response = sagemaker_runtime.invoke_endpoint_with_response_stream(
        EndpointName=endpoint_name,
        Body=json.dumps(request_body),
        ContentType="application/json",
    )
    generated_text = ""
    for line in LineIterator(response["Body"]):
        if line:
            content = parse_streaming_response(line.decode("utf-8"))
            if content:
                generated_text += content
                print(content, end="", flush=True)
    print()
    return generated_text

def invoke_teacher_model(prompt):
    """Invoke the teacher model (Qwen3-8B) for comparison."""
    request_body = {
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 1024,
        "temperature": 0.3,
    }
    response = sagemaker_runtime.invoke_endpoint(
        EndpointName=TEACHER_ENDPOINT,
        Body=json.dumps(request_body),
        ContentType="application/json",
    )
    result = json.loads(response["Body"].read().decode("utf-8"))
    return result["choices"][0]["message"]["content"]

print("Inference utilities ready.")

In [ ]:
# Test 1: Emergency case - SFT model vs Teacher model
test_prompt_1 = (
    "Assess the following patient and provide the likely condition, "
    "triage level, and treatment protocol.\n\n"
    "Patient ID: PT-001\nAge: 62\nSymptoms: chest pain, shortness of breath, sweating\n"
    "Reported Severity: high"
)

print("=" * 60)
print("Test 1: Emergency case - chest pain")
print("=" * 60)
print()
print("=" * 60)
print("SFT MODEL RESPONSE (Qwen2.5-7B fine-tuned):")
print("=" * 60)
sft_response = invoke_fine_tuned_model(test_prompt_1)
print()
print("=" * 60)
print("TEACHER MODEL RESPONSE (Qwen3-8B base):")
print("=" * 60)
teacher_response = invoke_teacher_model(test_prompt_1)
print(teacher_response)

In [ ]:
# Test 2: Non-urgent case - SFT model vs Teacher model
test_prompt_2 = (
    "Assess the following patient and provide the likely condition, "
    "triage level, and treatment protocol.\n\n"
    "Patient ID: PT-007\nAge: 29\nSymptoms: sneezing, runny nose, itchy eyes\n"
    "Reported Severity: low"
)

print("=" * 60)
print("Test 2: Non-urgent case - seasonal allergies")
print("=" * 60)
print()
print("=" * 60)
print("SFT MODEL RESPONSE (Qwen2.5-7B fine-tuned):")
print("=" * 60)
sft_response = invoke_fine_tuned_model(test_prompt_2)
print()
print("=" * 60)
print("TEACHER MODEL RESPONSE (Qwen3-8B base):")
print("=" * 60)
teacher_response = invoke_teacher_model(test_prompt_2)
print(teacher_response)

## 7. Cleanup

Uncomment and run the cells below to delete the deployed resources when you are done.

> **Note:** Delete resources in order: Model > Endpoint > Endpoint Config.

In [ ]:
# Delete model
# Model.get(model_name=model_name).delete()
# print(f"Deleted Model: {model_name}")

In [ ]:
# Delete endpoint
# Endpoint.get(endpoint_name=endpoint_name).delete()
# print(f"Deleted Endpoint: {endpoint_name}")

In [ ]:
# Delete endpoint config
# EndpointConfig.get(endpoint_config_name=endpoint_config_name).delete()
# print(f"Deleted EndpointConfig: {endpoint_config_name}")

In [ ]:
# Also delete the teacher model endpoint if no longer needed
# import sagemaker as sagemaker_v1
# teacher_predictor = sagemaker_v1.Predictor(endpoint_name="workshop-qwen3-8b", sagemaker_session=sagemaker_v1.Session())
# teacher_predictor.delete_endpoint()
# print("Deleted teacher endpoint: workshop-qwen3-8b")